In [9]:
# Standard libraries
import os
from typing import List, Tuple

# Data manipulation and numerical computation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn utilities
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV,cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.feature_selection import VarianceThreshold, mutual_info_classif, mutual_info_regression, f_classif, f_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from catboost import CatBoostRegressor

# Other utilities
from scipy.stats import randint, uniform
import joblib

# Standard libraries
from typing import List, Tuple

# Data manipulation and numerical computation
import numpy as np
import pandas as pd

# Scikit-learn utilities
from sklearn.impute import SimpleImputer

In [10]:
path = "../../data/processed/"
dfs_processed = {}

# read all dataframes and keep them in dfs
for parquet in os.listdir(path):  # List all files in the directory
    if parquet.endswith(".parquet"):
        name = parquet.split(".parquet")[0]  # Get name without extension
        dfs_processed[name] = pd.read_parquet(os.path.join(path, parquet))  # Use os.path.join for paths
        print(f"Loaded {name} with shape {dfs_processed[name].shape}")

sites = dfs_processed[list(dfs_processed.keys())[6]]

regiones = sites['HERlvl1Code'].drop_duplicates().tolist()
len(regiones)


Loaded 03_CLEAN_COMPLETE_DF with shape (49863, 174)
Loaded 03_CLEAN_COMPLETE_DF_02 with shape (49231, 623)
Loaded 03_COMPLETE_TEST with shape (43568, 3)
Loaded 03_COMPLETE_TRAIN with shape (49441, 2332)
Loaded 03_COMPLETE_TRAIN_2 with shape (49231, 2849)
Loaded clean_train with shape (43568, 4)
Loaded dep_codes with shape (49231, 12)
Loaded dep_test with shape (5063, 12)
Loaded taxones_pressure with shape (5663, 2333)
Loaded taxones_pressure_epm_predict with shape (5663, 2850)
Loaded taxones_pressure_epm_train with shape (43568, 2853)
Loaded taxones_pressure_predict with shape (5663, 2333)
Loaded taxones_pressure_train with shape (43568, 2336)


22

In [11]:
path = "../../notebooks/06_cb_regression/dfs_taxon_p_epm/"
dfs = {}

# read all dataframes and keep them in dfs
for parquet in os.listdir(path):  # List all files in the directory
    if parquet.endswith(".parquet"):
        name = parquet.split(".parquet")[0]  # Get name without extension
        dfs[name] = pd.read_parquet(os.path.join(path, parquet))  # Use os.path.join for paths
        print(f"Loaded {name} with shape {dfs[name].shape}")

from types import SimpleNamespace

# Después de llenar `dfs`:
d = SimpleNamespace(**dfs)

Loaded df_1 with shape (1485, 148)
Loaded df_10 with shape (3926, 504)
Loaded df_11 with shape (1289, 281)
Loaded df_12 with shape (4677, 434)
Loaded df_13 with shape (1003, 269)
Loaded df_14 with shape (7742, 329)
Loaded df_15 with shape (1223, 434)
Loaded df_16 with shape (379, 336)
Loaded df_17 with shape (803, 246)
Loaded df_18 with shape (1164, 447)
Loaded df_19 with shape (500, 220)
Loaded df_2 with shape (352, 360)
Loaded df_20 with shape (508, 364)
Loaded df_21 with shape (2663, 298)
Loaded df_22 with shape (152, 434)
Loaded df_3 with shape (4996, 217)
Loaded df_4 with shape (596, 442)
Loaded df_5 with shape (2313, 411)
Loaded df_6 with shape (2134, 423)
Loaded df_7 with shape (524, 352)
Loaded df_8 with shape (548, 232)
Loaded df_9 with shape (10254, 516)


In [12]:

def train_catboost_region(
    cleandf: pd.DataFrame,
    target: str = 'IBD',
    test_size: float = 0.20,
    random_state: int = 42,
    early_stopping_rounds: int = 200,
    cat_params: dict | None = None,
):
    """
    Entrena CatBoost para una región usando cleandf.
    - Separa train (con target) y score (sin target)
    - Preprocesa (imputación num/cat + OHE)
    - Entrena con early stopping
    - Regresa: modelo (Pipeline), métricas, scored_df (filas sin target con predicción)
    """
    # 1) separar train / score
    df_train = cleandf[cleandf[target].notna()].copy()
    df_score = cleandf[cleandf[target].isna()].copy()

    # X / y
    drop_cols = [c for c in ['IBD','IBD_EQR','IBD_EQR_Status'] if c in cleandf.columns]
    X = df_train.drop(columns=drop_cols, errors='ignore')
    y = df_train[target].astype(float)

    # split
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=test_size, random_state=random_state)

    # 2) columnas num/cat
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

    # preprocesamiento
    pre = ColumnTransformer([
        ('num', Pipeline([
            ('imp', SimpleImputer(strategy='median')),
        ]), num_cols),
        ('cat', Pipeline([
            ('imp', SimpleImputer(strategy='constant', fill_value='(missing)')),
            ('ohe', OneHotEncoder(handle_unknown='ignore')),
        ]), cat_cols)
    ])

    # 3) modelo CatBoost (parámetros por defecto + override opcional)
    base_params = dict(
        depth=6, learning_rate=0.05, n_estimators=3000,
        loss_function='RMSE', random_state=random_state, verbose=0
    )
    if cat_params:
        base_params.update(cat_params)

    cb = Pipeline([
        ('pre', pre),
        ('model', CatBoostRegressor(**base_params))
    ])

    # 4) ajustar: primero ajustamos el preprocesador para armar matrices, luego el modelo con early stopping
    Xtr_proc = pre.fit_transform(X_tr)
    Xte_proc = pre.transform(X_te)
    cb.named_steps['model'].fit(
        Xtr_proc, y_tr,
        eval_set=(Xte_proc, y_te),
        use_best_model=True,
        early_stopping_rounds=early_stopping_rounds
    )

    # 5) métricas en hold-out y train (usando el pipeline para que transforme igual)
    r2_tr  = cb.score(X_tr, y_tr)
    r2_te  = cb.score(X_te, y_te)
    pred_tr = cb.predict(X_tr); pred_te = cb.predict(X_te)
    metrics = {
        'R2_train': r2_tr,
        'R2_valid': r2_te,
        'MAE_train': mean_absolute_error(y_tr, pred_tr),
        'MAE_valid': mean_absolute_error(y_te, pred_te),
        'RMSE_train': mean_squared_error(y_tr, pred_tr),
        'RMSE_valid': mean_squared_error(y_te, pred_te),
        'best_iterations': int(cb.named_steps['model'].get_best_iteration() or base_params['n_estimators'])
    }

    # 6) predicciones para filas sin target de esta región (si existen)
    if not df_score.empty:
        X_score = df_score.drop(columns=drop_cols, errors='ignore')
        df_score[target + '_pred'] = cb.predict(X_score)
        scored_df = df_score
        
    else:
        scored_df = pd.DataFrame(columns=list(cleandf.columns) + [target + '_pred'])

    return cb, metrics, scored_df

In [14]:
predicciones = []
all_metrics = {}  # guardará {region: métricas}

for region in regiones:
    attr = f"df_{int(region)}"           # df_1, df_2, df_18, ...
    if not hasattr(d, attr):
        # fallback por si tus nombres vienen con ceros: df_01
        attr = f"df_{str(region)}"
    if not hasattr(d, attr):
        raise AttributeError(f"No encontré {attr} en d")

    cleandf = getattr(d, attr)
    model, m, scored = train_catboost_region(cleandf, target='IBD')  # m = métricas de esa región
    all_metrics[region] = m

    # Mantener índice (SamplingOperations_code) y solo la predicción
    solo_ibd = scored[['IBD_pred']].copy()
    solo_ibd['region'] = region
    predicciones.append(solo_ibd)

predicciones = pd.concat(predicciones, axis=0)  # índice preservado
predicciones.index.name = 'SamplingOperations_code'

metrics_df = (pd.DataFrame.from_dict(all_metrics, orient='index')
                .reset_index()
                .rename(columns={'index':'region'}))
metrics_df


KeyboardInterrupt: 

In [ ]:
metrics_df.to_csv('metrisc2.csv',index=False)